# 08 — Batch Mode & Multi-Device Synchronization

This notebook covers advanced patterns for coordinating multiple Ivium devices simultaneously:

- **Status parameter** — a global integer shared between Python and IviumSoft used as a synchronization signal
- **Multi-device setpoints** — apply potential or current to multiple devices in one call without switching instances
- **Parallel experiment patterns** — start and monitor measurements on all devices concurrently

### Typical use cases

- Trigger all devices to start a measurement at the same time
- Have IviumSoft-side scripts wait for a Python signal before proceeding
- Apply the same setpoint to multiple devices simultaneously
- Run independent experiments in parallel and collect all results

### Prerequisites
- Multiple IviumSoft instances running (or one Ivium-n-Soft with multiple channels)
- At least two devices connected for multi-device sections

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
from pyvium import Pyvium

Pyvium.open_driver()
print(f"Active instances: {Pyvium.get_active_iviumsoft_instances()}")

## 1. The Status Parameter

`set_status_par(value)` / `get_status_par()` expose a global integer shared between Python and IviumSoft. It is per-instance — each IviumSoft window has its own status parameter.

### Synchronization convention

There is no fixed meaning for the values — you choose the convention. A common one:

| Value | Meaning |
|-------|--------|
| `0` | Idle / waiting |
| `1` | Python signals "ready to start" |
| `2` | IviumSoft signals "measurement running" |
| `3` | IviumSoft signals "measurement done" |
| `-1` | Abort / error |

In [ ]:
Pyvium.select_iviumsoft_instance(1)

# Write a value
Pyvium.set_status_par(0)
print(f"Set status par to 0")

# Read it back
val = Pyvium.get_status_par()
print(f"Get status par: {val}")

## 2. Python → IviumSoft Trigger

Pattern: Python sets the status parameter to `1` to signal IviumSoft to start a method. IviumSoft-side scripting polls the status parameter and begins when it sees `1`.

This lets IviumSoft-side automation react to Python events.

In [ ]:
# Signal IviumSoft to proceed
Pyvium.select_iviumsoft_instance(1)
Pyvium.set_status_par(1)  # "go"
print("Triggered IviumSoft instance 1 (status par = 1)")

## 3. IviumSoft → Python Completion Signal

Pattern: IviumSoft sets the status parameter to `3` when a measurement finishes. Python polls it and proceeds when it sees `3`.

In [ ]:
def wait_for_status(instance: int, target_value: int, timeout_s: float = 120.0):
    """Block until the given instance's status par reaches target_value, or timeout."""
    Pyvium.select_iviumsoft_instance(instance)
    t_start = time.time()

    while True:
        current = Pyvium.get_status_par()
        if current == target_value:
            return True
        if time.time() - t_start > timeout_s:
            print(f"Timeout waiting for instance {instance} status={target_value}")
            return False
        time.sleep(0.2)

# Example: wait up to 60 s for IviumSoft to set status par to 3 ("done")
# done = wait_for_status(instance=1, target_value=3, timeout_s=60.0)
# if done:
#     print("Measurement done")
print("wait_for_status() helper defined")

## 4. Triggering All Instances Simultaneously

By quickly looping through all instances and setting the status parameter, you can trigger all devices near-simultaneously.

In [ ]:
def broadcast_status(value: int):
    """Set the status parameter on all active IviumSoft instances."""
    instances = Pyvium.get_active_iviumsoft_instances()
    for inst in instances:
        Pyvium.select_iviumsoft_instance(inst)
        Pyvium.set_status_par(value)
    print(f"Broadcast status={value} to instances {instances}")

broadcast_status(0)   # reset all to idle
time.sleep(1.0)
broadcast_status(1)   # trigger all

## 5. Multi-Device Setpoints

`set_device_potential(instance, V)` and `set_device_current(instance, A)` apply a setpoint to a specific device instance **without changing the currently selected instance**. This allows rapid setpoint updates across all devices without the overhead of instance-switching.

- `instance`: the IviumSoft instance number (1-based)
- Both functions require the driver to be open but **not** the target device to be the currently selected instance

In [ ]:
# Apply the same potential to all active instances simultaneously
HOLD_POTENTIAL = 0.1  # V

instances = Pyvium.get_active_iviumsoft_instances()
for inst in instances:
    Pyvium.set_device_potential(inst, HOLD_POTENTIAL)
    print(f"Instance {inst}: potential → {HOLD_POTENTIAL} V")

print("All devices set")

In [ ]:
# Apply a gradient of potentials across devices
potentials = [0.0, 0.1, 0.2, 0.3]  # V — one per instance

for inst, V in zip(instances, potentials):
    Pyvium.set_device_potential(inst, V)
    print(f"Instance {inst}: {V:.2f} V")

## 6. Running Methods on All Devices in Parallel

Start a method on each device, then collect data from all of them after all complete.

In [ ]:
METHOD_PATH     = r"C:/IviumStat/AppMethods/LSV.imf"
EXPECTED_POINTS = 160
POLL_INTERVAL   = 0.5

def start_all(instances, method_path):
    for inst in instances:
        Pyvium.select_iviumsoft_instance(inst)
        Pyvium.connect_device()
        Pyvium.start_method(method_path)
        print(f"Instance {inst}: method started")

def wait_all(instances, expected_points):
    remaining = set(instances)
    while remaining:
        finished = set()
        for inst in remaining:
            Pyvium.select_iviumsoft_instance(inst)
            n = Pyvium.get_available_data_points_number()
            if n >= expected_points:
                finished.add(inst)
        for inst in finished:
            print(f"  Instance {inst}: complete")
        remaining -= finished
        if remaining:
            time.sleep(POLL_INTERVAL)
    print("All measurements complete")

def collect_all(instances):
    results = {}
    for inst in instances:
        Pyvium.select_iviumsoft_instance(inst)
        n = Pyvium.get_available_data_points_number()
        rows = [Pyvium.get_data_point(i) for i in range(1, n + 1)]
        results[inst] = pd.DataFrame(rows, columns=["E (V)", "I (A)", "aux"])
        print(f"  Instance {inst}: {len(rows)} points collected")
    return results

print("Helpers defined. Uncomment below to run:")
# start_all(instances, METHOD_PATH)
# wait_all(instances, EXPECTED_POINTS)
# data = collect_all(instances)

## 7. Plotting Multi-Device Results

In [ ]:
# Assumes `data` dict was populated by collect_all() above
# data = collect_all(instances)

# Synthetic example for illustration
import numpy as np
data = {
    1: pd.DataFrame({"E (V)": np.linspace(0, 1, 100),
                     "I (A)": np.random.randn(100) * 1e-6 + 1e-5}),
    2: pd.DataFrame({"E (V)": np.linspace(0, 1, 100),
                     "I (A)": np.random.randn(100) * 1e-6 + 2e-5}),
}

fig, ax = plt.subplots(figsize=(8, 5))
for inst, df in data.items():
    ax.plot(df["E (V)"], df["I (A)"] * 1e6, label=f"Device {inst}")

ax.set_xlabel("Potential (V)")
ax.set_ylabel("Current (µA)")
ax.set_title("Parallel LSV — All Devices")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Complete Synchronized Workflow

Full template: connect all devices, run a method in parallel, collect data, save and plot.

In [ ]:
import os

METHOD_PATH    = r"C:/IviumStat/AppMethods/LSV.imf"
OUTPUT_DIR     = r"C:/IviumStat/Data"
EXPECTED_PTS   = 160

try:
    Pyvium.open_driver()
    instances = Pyvium.get_active_iviumsoft_instances()
    print(f"Devices: {instances}")

    # Step 1: connect all and start
    for inst in instances:
        Pyvium.select_iviumsoft_instance(inst)
        Pyvium.connect_device()

    for inst in instances:
        Pyvium.select_iviumsoft_instance(inst)
        Pyvium.start_method(METHOD_PATH)
        print(f"  Instance {inst}: started")

    # Step 2: wait for all to finish
    remaining = set(instances)
    while remaining:
        done = set()
        for inst in remaining:
            Pyvium.select_iviumsoft_instance(inst)
            if Pyvium.get_available_data_points_number() >= EXPECTED_PTS:
                done.add(inst)
        remaining -= done
        if remaining:
            time.sleep(0.5)
    print("All done")

    # Step 3: collect and save
    for inst in instances:
        Pyvium.select_iviumsoft_instance(inst)
        out = os.path.join(OUTPUT_DIR, f"device_{inst}.idf")
        if os.path.isdir(OUTPUT_DIR):
            Pyvium.save_data(out)
            print(f"  Instance {inst} → {out}")

finally:
    try:
        Pyvium.close_driver()
    except Exception:
        pass

---

## Summary

### Status parameter (synchronization)

| Task | Method |
|------|--------|
| Write status to selected instance | `set_status_par(value)` |
| Read status from selected instance | `get_status_par()` → int |
| Broadcast to all | Loop `select_iviumsoft_instance(n)` + `set_status_par(v)` |

### Multi-device setpoints

| Task | Method | Notes |
|------|--------|-------|
| Set potential on any instance | `set_device_potential(instance, V)` | No instance switch needed |
| Set current on any instance | `set_device_current(instance, A)` | Galvanostatic mode |

### Parallel measurement pattern

```python
# 1. Start all
for inst in instances:
    Pyvium.select_iviumsoft_instance(inst)
    Pyvium.start_method(path)

# 2. Wait for all
while not all_done(instances):
    time.sleep(0.5)

# 3. Collect from all
for inst in instances:
    Pyvium.select_iviumsoft_instance(inst)
    data[inst] = read_all_points()
```

---

## Notebook Series Complete

You've reached the end of the Pyvium documentation series. Here's the full map:

| Notebook | Topic | Hardware needed? |
|----------|-------|:-:|
| 01 | Getting started, error handling | No (just IviumSoft) |
| 02 | Device & instance management | Yes |
| 03 | Direct mode — potentiostat basics | Yes |
| 04 | Direct mode — traces, DAC/ADC, digital I/O, AC | Yes |
| 05 | BiStat (WE2) and WE32 multichannel | Yes (+ accessories) |
| 06 | Method mode — run experiments, read data | Yes |
| 07 | IDF file parsing and CSV export | **No** |
| 08 | Batch mode & multi-device synchronization | Yes (multiple devices) |